<a href="https://colab.research.google.com/github/beingAtharun/CN-CD-LAB/blob/main/gemma%20model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
%pip -q install --upgrade "transformers>=4.51.3" accelerate sentencepiece einops
%pip -q install bitsandbytes
# Colab usually has a recent torch. If you need to pin/upgrade, run this too:
# %pip -q install "torch>=2.4.0"

In [12]:
tok = userdata.get("HF_TOKEN")
tok = tok.strip() if tok else tok

In [13]:
from google.colab import userdata

print("has(HF_TOKEN):", userdata.has("HF_TOKEN"))
tok = userdata.get("HF_TOKEN")
print("is None:", tok is None)
print("len:", len(tok) if tok else None)
print("starts with hf_:", bool(tok and tok.strip().startswith("hf_")))

AttributeError: module 'google.colab.userdata' has no attribute 'has'

In [14]:
from google.colab import userdata

tok = userdata.get("HF_TOKEN")   # returns None if not set or no notebook access
print("HF_TOKEN present?:", tok is not None)
print("Length:", len(tok) if tok else None)
print("Looks like an HF token?:", bool(tok and tok.strip().startswith("hf_")))

HF_TOKEN present?: True
Length: 37
Looks like an HF token?: True


In [15]:
from huggingface_hub import login, logout, whoami
from google.colab import userdata

# Clear any cached/invalid credentials (safe to call even if none exist)
try:
    logout()
except Exception:
    pass

hf_token = userdata.get("HF_TOKEN")
hf_token = hf_token.strip()
assert hf_token and hf_token.startswith("hf_"), "HF_TOKEN looks missing or malformed."

login(token=hf_token, add_to_git_credential=False)

# Sanity: confirm the account Colab is using
print("HF identity:", whoami().get("name", "<ok>"))

Not logged in!


HF identity: venteja


In [16]:
from huggingface_hub import login, logout, whoami
from google.colab import userdata

# Clear any cached/invalid credentials
try:
    logout()
except Exception:
    pass

hf_token = userdata.get("HF_TOKEN")
hf_token = hf_token.strip() if hf_token else None
assert hf_token and hf_token.startswith("hf_"), "HF_TOKEN missing or malformed. Re-add it in the Secrets panel."

login(token=hf_token, add_to_git_credential=False)

print("HF identity:", whoami().get("name", "<ok>"))

HF identity: venteja


In [17]:
%pip -q install --upgrade "transformers>=4.51.3" accelerate sentencepiece einops
%pip -q install bitsandbytes

In [18]:
from huggingface_hub import login, logout, whoami
from google.colab import userdata

# Clear any cached/invalid credentials
try:
    logout()
except Exception:
    pass

hf_token = userdata.get("HF_TOKEN")
hf_token = hf_token.strip() if hf_token else None
assert hf_token and hf_token.startswith("hf_"), "HF_TOKEN missing or malformed. Re-add it in the Secrets panel."

login(token=hf_token, add_to_git_credential=False)
print("HF identity:", whoami().get("name", "<ok>"))

print("\nIf loading Gemma returns 401, open https://huggingface.co/google/gemma-3-4b-it "
      "and click 'Acknowledge license' while logged into the same account.")

HF identity: venteja

If loading Gemma returns 401, open https://huggingface.co/google/gemma-3-4b-it and click 'Acknowledge license' while logged into the same account.


In [19]:
import torch
from transformers import AutoProcessor, Gemma3ForConditionalGeneration, BitsAndBytesConfig

assert torch.cuda.is_available(), "CUDA not found. In Colab: Runtime → Change runtime type → GPU (T4)."
print("GPU:", torch.cuda.get_device_name(0))

MODEL_ID = "google/gemma-3-4b-it"

# 4-bit quantization (T4-friendly)
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.bfloat16)

# perf niceties on T4
torch.backends.cuda.matmul.allow_tf32 = True
torch.set_float32_matmul_precision("high")

processor = AutoProcessor.from_pretrained(MODEL_ID)  # uses your logged-in token
model = Gemma3ForConditionalGeneration.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    quantization_config=bnb,
    attn_implementation="sdpa",
)

print("Gemma loaded ✅")

GPU: Tesla T4


processor_config.json:   0%|          | 0.00/70.0 [00:00<?, ?B/s]

chat_template.json:   0%|          | 0.00/1.61k [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

The image processor of type `Gemma3ImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


...

model.safetensors.index.json:   0%|          | 0.00/90.6k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

Gemma loaded ✅


In [20]:
import json, re, ast
from typing import Any

def extract_first_json(text: str) -> str:
    try:
        json.loads(text); return text
    except Exception:
        pass
    t = re.sub(r"^```(json)?", "", text.strip(), flags=re.IGNORECASE)
    t = re.sub(r"```$", "", t.strip())
    start = t.find("{"); end = t.rfind("}")
    if start != -1 and end != -1:
        return t[start:end+1]
    start_b = t.find("["); end_b = t.rfind("]")
    if start_b != -1 and end_b != -1:
        return t[start_b:end_b+1]
    raise ValueError("Could not extract a JSON-like region")

def _basic_json_fixes(s: str) -> str:
    s = s.replace("“","\"").replace("”","\"").replace("’","'").replace("‘","'")
    s = re.sub(r"//.*?$", "", s, flags=re.MULTILINE)
    s = re.sub(r"/\*.*?\*/", "", s, flags=re.DOTALL)
    s = re.sub(r",\s*([\]}])", r"\1", s)
    s = re.sub(r"\bTrue\b", "true", s)
    s = re.sub(r"\bFalse\b", "false", s)
    s = re.sub(r"\bNone\b", "null", s)
    return s

def safe_json_loads(text: str) -> Any:
    try:
        return json.loads(text)
    except Exception:
        pass
    try:
        frag = extract_first_json(text)
        try:
            return json.loads(frag)
        except Exception:
            pass
        fixed = _basic_json_fixes(frag)
        try:
            return json.loads(fixed)
        except Exception:
            pass
        obj = ast.literal_eval(fixed)
        return json.loads(json.dumps(obj))
    except Exception:
        raise json.JSONDecodeError("Could not parse model output as JSON", text, 0)

In [21]:
from PIL import Image
from typing import Dict

def vlm_infer_json_gemma(
    image: Image.Image,
    prompt: str,
    temperature: float = 0.0,
    max_new_tokens: int = 512,
    pan_and_scan: bool = True
) -> Dict:
    messages = [
        {"role": "system",
         "content": [{"type":"text","text":"Return ONLY a single valid JSON object for the user's request. No extra text."}]},
        {"role": "user",
         "content": [{"type":"image","image": image}, {"type":"text","text": prompt}]},
    ]

    # Gemma‑3 chat template; pan&scan helps tall/wide/high‑res shelves.
    chat_text = processor.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        do_pan_and_scan=pan_and_scan
    )

    inputs = processor(text=[chat_text], images=[image], return_tensors="pt").to("cuda")

    with torch.no_grad():
        out_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            do_sample=(temperature > 0),
            cache_implementation="static",
        )

    gen_text = processor.batch_decode(
        out_ids[:, inputs["input_ids"].shape[-1]:],
        skip_special_tokens=True
    )[0].strip()

    try:
        return safe_json_loads(gen_text)
    except Exception:
        # One-shot repair (no image resend)
        repair_prompt = "Your previous output was not valid JSON. Return ONLY valid JSON for the same task. No comments, no extra text."
        messages2 = [
            {"role":"system","content":[{"type":"text","text":"Return ONLY valid JSON. No extra text."}]},
            {"role":"user","content":[{"type":"text","text":repair_prompt}, {"type":"text","text":prompt}]},
        ]
        chat_text2 = processor.apply_chat_template(messages2, tokenize=False, add_generation_prompt=True)
        inputs2 = processor(text=[chat_text2], return_tensors="pt").to("cuda")
        with torch.no_grad():
            out_ids2 = model.generate(**inputs2, max_new_tokens=max_new_tokens, temperature=0.0, do_sample=False)
        gen_text2 = processor.batch_decode(
            out_ids2[:, inputs2["input_ids"].shape[-1]:], skip_special_tokens=True
        )[0].strip()
        return safe_json_loads(gen_text2)

In [22]:
PROMPT = """You are an automated robotic shelf inspection system.

Input: A image of a pop shop rack containing multiple shelves with food and drink items.

Your tasks:

1. Product Categorization
   Detect all visible items and group them into meaningful product categories.
   Examples: "chips", "soft drinks", "milk", "biscuits", "chocolates", etc.
   Return each category name only once.

2. Empty Shelf Detection
   Identify shelves that contain no products at all.
   Return the total number of empty shelves as an integer.

3. Misplaced Item Detection
   If a shelf contains one or more items that do NOT belong to the dominant product category of that shelf, treat them as misplaced items.
   For each misplaced item, return:
   - product name
   - bounding box in image coordinates: {x_min, y_min, x_max, y_max}

Output Rules
Return ONLY valid JSON in the exact structure below. No extra text.

{
 "product_categories": ["", "", ""],
 "empty_shelves": 0,
 "misplaced_items": [
   {
     "product_name": "",
     "bounding_box": { "x_min": 0, "y_min": 0, "x_max": 0, "y_max": 0 }
   }
 ]
}
"""

TILE_PROMPT = """You are analyzing a cropped tile of a larger retail shelf photo.
List all visible products in THIS TILE ONLY.

Return JSON with:
{
  "products": [
    {
      "product_name": "",
      "category": "",
      "bounding_box": { "x_min": 0, "y_min": 0, "x_max": 0, "y_max": 0 }  // tile pixel coords or [0..1]
    }
  ]
}

Rules:
- Only include items visible within this tile.
- If unsure of product_name, use a generic name like "item".
- Bounding boxes must tightly cover the visible extent of the item.
- Return ONLY valid JSON. No explanations.
"""

In [23]:
from typing import Dict, Any
import math
from PIL import Image, ImageDraw, ImageFont

def iou(boxA, boxB):
    xA = max(boxA[0], boxB[0]); yA = max(boxA[1], boxB[1])
    xB = min(boxA[2], boxB[2]); yB = min(boxA[3], boxB[3])
    inter = max(0, xB-xA) * max(0, yB-yA)
    if inter <= 0: return 0.0
    areaA = max(0, boxA[2]-boxA[0]) * max(0, boxA[3]-boxA[1])
    areaB = max(0, boxB[2]-boxB[0]) * max(0, boxB[3]-boxB[1])
    union = areaA + areaB - inter
    return inter / max(union, 1e-6)

def nms_merge(instances, iou_thresh=0.5):
    instances = sorted(instances, key=lambda d: (d["box_px"][2]-d["box_px"][0])*(d["box_px"][3]-d["box_px"][1]), reverse=True)
    kept = []
    for d in instances:
        if all(iou(d["box_px"], k["box_px"]) < iou_thresh for k in kept):
            kept.append(d)
    return kept

def clip_box(x1,y1,x2,y2,W,H):
    x1 = max(0, min(W-1, int(round(x1))))
    y1 = max(0, min(H-1, int(round(y1))))
    x2 = max(0, min(W-1, int(round(x2))))
    y2 = max(0, min(H-1, int(round(y2))))
    if x2 < x1: x1,x2 = x2,x1
    if y2 < y1: y1,y2 = y2,y1
    return [x1,y1,x2,y2]

def to_1000(box_px, W, H):
    x1,y1,x2,y2 = box_px
    X1 = max(0, min(1000, int(round((x1/W)*1000))))
    Y1 = max(0, min(1000, int(round((y1/H)*1000))))
    X2 = max(0, min(1000, int(round((x2/W)*1000))))
    Y2 = max(0, min(1000, int(round((y2/H)*1000))))
    if X2 < X1: X1,X2 = X2,X1
    if Y2 < Y1: Y1,Y2 = Y2,Y1
    return [X1,Y1,X2,Y2]

def from_tile_box_to_full(bb, x0,y0,x1,y1):
    if not (isinstance(bb,(list,tuple)) and len(bb)==4):
        return None
    bx1, by1, bx2, by2 = bb
    tw, th = (x1-x0), (y1-y0)
    if all(isinstance(v,(int,float)) for v in bb) and all(0 <= v <= 1.2 for v in bb):
        fx1 = x0 + bx1 * tw; fy1 = y0 + by1 * th
        fx2 = x0 + bx2 * tw; fy2 = y0 + by2 * th
    elif all(isinstance(v,(int,float)) for v in bb) and all(0 <= v <= 1000 for v in bb):
        fx1 = x0 + (bx1/1000) * tw; fy1 = y0 + (by1/1000) * th
        fx2 = x0 + (bx2/1000) * tw; fy2 = y0 + (by2/1000) * th
    else:
        fx1 = x0 + bx1; fy1 = y0 + by1
        fx2 = x0 + bx2; fy2 = y0 + by2
    return [fx1, fy1, fx2, fy2]

def run_tiled(
    image: Image.Image,
    global_prompt: str,
    tile_prompt: str,
    grid=(4,3),
    overlap=0.2,
    temperature=0.0,
    max_new_tokens=1024,
    pan_and_scan=True
) -> Dict[str, Any]:
    W, H = image.size
    gx, gy = grid
    tile_w = math.ceil(W / gx)
    tile_h = math.ceil(H / gy)
    ox = int(overlap * tile_w)
    oy = int(overlap * tile_h)

    all_instances = []
    for yi in range(gy):
        for xi in range(gx):
            x0 = max(0, xi*tile_w - (ox if xi>0 else 0))
            y0 = max(0, yi*tile_h - (oy if yi>0 else 0))
            x1 = min(W, (xi+1)*tile_w + (ox if xi<gx-1 else 0))
            y1 = min(H, (yi+1)*tile_h + (oy if yi<gy-1 else 0))
            crop = image.crop((x0,y0,x1,y1))

            try:
                partial = vlm_infer_json_gemma(
                    crop, tile_prompt,
                    temperature=temperature,
                    max_new_tokens=max_new_tokens,
                    pan_and_scan=pan_and_scan
                )
            except Exception:
                continue

            for p in partial.get("products", []):
                name = (p.get("product_name") or "").strip() or "item"
                cat  = (p.get("category") or "").strip()
                bb   = p.get("bounding_box", [0,0,0,0])
                if isinstance(bb, dict):
                    bb = [bb.get("x_min",0), bb.get("y_min",0), bb.get("x_max",0), bb.get("y_max",0)]
                full = from_tile_box_to_full(bb, x0,y0,x1,y1)
                if full is None:
                    continue
                box_px = clip_box(*full, W, H)
                if (box_px[2]-box_px[0]) * (box_px[3]-box_px[1]) < 0.0003 * (W*H):
                    continue
                all_instances.append({"product_name": name, "category": cat, "box_px": box_px})

    merged = nms_merge(all_instances, iou_thresh=0.5)

    try:
        global_result = vlm_infer_json_gemma(
            image, global_prompt,
            temperature=temperature,
            max_new_tokens=max_new_tokens,
            pan_and_scan=pan_and_scan
        )
        cats = sorted(set(global_result.get("product_categories", [])))
        empty_shelves = int(global_result.get("empty_shelves", 0))
    except Exception:
        cats, empty_shelves = [], 0

    final_products = [{
        "product_name": inst.get("product_name","item"),
        "category":     inst.get("category",""),
        "bounding_box": to_1000(inst["box_px"], W, H)
    } for inst in merged]

    return {
        "product_categories": cats,
        "empty_shelves": empty_shelves,
        "products": final_products
    }

In [24]:
from google.colab import files
uploaded = files.upload()  # If Edge picker fails, use left Files panel → Upload
img_path = list(uploaded.keys())[0]

image = Image.open(img_path).convert("RGB")

result = run_tiled(
    image,
    global_prompt=PROMPT,
    tile_prompt=TILE_PROMPT,
    grid=(4,3),
    overlap=0.2,
    temperature=0.0,
    max_new_tokens=1024,
    pan_and_scan=True
)

print(json.dumps(result, indent=2, ensure_ascii=False))

# Visualization
W, H = image.size
vis = image.copy()
draw = ImageDraw.Draw(vis)
try:
    font = ImageFont.truetype("DejaVuSans.ttf", size=max(12, W//85))
except:
    font = None

for p in result.get("products", []):
    X1,Y1,X2,Y2 = p["bounding_box"]
    x1 = int(round((X1/1000)*W)); y1 = int(round((Y1/1000)*H))
    x2 = int(round((X2/1000)*W)); y2 = int(round((Y2/1000)*H))
    draw.rectangle([x1,y1,x2,y2], outline="red", width=3)
    label = f"{p.get('product_name','item')} ({p.get('category','')})"
    try:
        draw.text((x1, max(0, y1-18)), label, fill="yellow",
                  font=font, stroke_width=2, stroke_fill="black")
    except:
        draw.text((x1, max(0, y1-18)), label, fill="yellow")

import os
out_path = os.path.splitext(img_path)[0] + "_gemma3_t4_viz.jpg"
vis.save(out_path)
print(f"Saved visualization to: {out_path}")

Saving image (2).png to image (2).png


The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


{
  "product_categories": [
    "biscuits",
    "chips",
    "milk",
    "soft drinks"
  ],
  "empty_shelves": 0,
  "products": [
    {
      "product_name": "Drink Tin Can",
      "category": "Beverage",
      "bounding_box": [
        203,
        269,
        551,
        737
      ]
    },
    {
      "product_name": "Tin Can",
      "category": "Food",
      "bounding_box": [
        454,
        269,
        802,
        737
      ]
    },
    {
      "product_name": "Chips",
      "category": "Food",
      "bounding_box": [
        203,
        0,
        551,
        402
      ]
    },
    {
      "product_name": "chips",
      "category": "snacks",
      "bounding_box": [
        454,
        0,
        802,
        402
      ]
    },
    {
      "product_name": "salsa",
      "category": "food",
      "bounding_box": [
        0,
        0,
        300,
        402
      ]
    },
    {
      "product_name": "Soda",
      "category": "Beverages",
      "bounding_box": [
      